# Satellite-to-Chip QKD Digital Twin (M5)

This notebook runs the PhotonTrust M5 satellite-chain scenarios and compares key accumulation across detector choices.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt

from photonstrust.config import load_config
from photonstrust.pipeline.satellite_chain import run_satellite_chain

In [ ]:
repo_root = Path.cwd()
configs = {
    "berlin_apd": repo_root / "configs" / "satellite" / "eagle1_analog_berlin.yml",
    "berlin_snspd": repo_root / "configs" / "satellite" / "eagle1_analog_snspd.yml",
    "micius_analog": repo_root / "configs" / "satellite" / "micius_analog.yml",
}

results = {}
for name, cfg_path in configs.items():
    cfg = load_config(cfg_path)
    out = repo_root / "results" / "notebook_satellite_chain" / name
    run = run_satellite_chain(cfg, output_dir=out)
    cert = run["certificate"]
    results[name] = {
        "decision": cert["signoff"]["decision"],
        "key_bits_accumulated": cert["pass"]["key_bits_accumulated"],
        "mean_key_rate_bps": cert["pass"]["mean_key_rate_bps"],
        "key_mbits_per_year": cert.get("annual_estimate", {}).get("key_mbits_per_year", 0.0),
        "certificate_path": run.get("output_path"),
    }

print(json.dumps(results, indent=2))

In [ ]:
labels = list(results.keys())
key_bits = [results[k]["key_bits_accumulated"] for k in labels]
yearly = [results[k]["key_mbits_per_year"] for k in labels]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(labels, key_bits)
axes[0].set_title("Key Bits Per Pass")
axes[0].set_ylabel("bits")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(labels, yearly)
axes[1].set_title("Estimated Annual Key Yield")
axes[1].set_ylabel("Mbit/year")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()